In [1]:
from datetime import date
from collections import Counter, defaultdict
import warnings

import periodictable
import h5py
import numpy as np

import qcportal
from qcportal.external import scaffold
from qcportal.molecules import Molecule
from qcportal.singlepoint import SinglepointDriver, QCSpecification
from qcportal.optimization import OptimizationSpecification
from qcelemental.models.procedures import OptimizationProtocols
from qcelemental.physical_constants import constants

ADDRESS = "https://api.qcarchive.molssi.org:443"
#qc_client = qcportal.PortalClient(ADDRESS, cache_dir=".")
from qcfractal.snowflake import FractalSnowflake
snowflake = FractalSnowflake()
client = snowflake.client()

/Users/jenniferclark/mamba/envs/qca-clean/lib/python3.11/site-packages/qcelemental/models/procedures.py:8: FutureWarning: qcelemental.models.procedures should be accessed through qcelemental.models (or qcelemental.models.v1 or .v2 for fixed QCSchema version). The 'models.procedures' route will be removed as soon as v0.70.0.
  warn(
Detected locale-related PostgreSQL startup failure; retrying pg_ctl start with LC_ALL=C


In [ ]:
#!aria2c "https://zenodo.org/records/19372923/files/architector_filtered.hdf5.gz?download=1"

## Helper Functions

In [3]:
def remove_extraneous_dimension(array):
    shape = list(np.shape(array))
    if 1 in shape:
        shape.remove(1)
    return np.array(array).reshape(shape)

def get_symbols(atomic_numbers):
    return [str(periodictable.elements[x])for x in remove_extraneous_dimension(atomic_numbers)]

def get_molecular_formula(atomic_numbers):
    return "".join([str(y) for x1, x2 in Counter(get_symbols(atomic_numbers)).items() for y in [x1, x2] if y != 1])

def get_molecular_weight(atomic_numbers):
    return sum(periodictable.elements[x].mass for x in remove_extraneous_dimension(atomic_numbers))


In [4]:
def apply_mapping(mapping, input_dict, index=0):

    output = defaultdict(dict)
    for key, value in mapping.items():
        if isinstance(value, str):
            data = input_dict[value]
            if not isinstance(data, str):
                if key == "geometry": # update number of frames
                    lx = np.shape(data)[0]
                
                if key != "geometry":
                    data = remove_extraneous_dimension(data)
                    lx = len(data)

                if lx is not None: # and len(np.shape(data)) > 1:
                    if len(data) == lx:
                        output[key] = data[index]
                        continue
                    else:
                        raise ValueError(f"Expected {lx} configuration, but {len(data)} are present")
            output[key] = data
        elif isinstance(value, tuple): # function, input pairs
            output[key] = value[0](*(input_dict[k2] for k2 in value[1:]))
        elif isinstance(value, list):
            output[key].update({k2: input_dict[k2] for k2 in value})
        elif isinstance(value, dict):
            output[key].update(apply_mapping(value, input_dict, index=0))
            
    return output
            
def convert_hdf5_group(hdf5_group):
    output = {}
    for key, value in hdf5_group.items():
        if isinstance(value, h5py.Group):
            output[key] = convert_hdf5_group(value)
        elif isinstance(value, h5py.Dataset):
            data = value[()]
            if isinstance(data, np.ndarray):
                output[key] = data
            elif isinstance(data, np.bytes_):
                output[key] = data.decode('utf-8')  # Convert to string
            else:
                output[key] = data.item() if isinstance(data, np.generic) else data  # Convert NumPy scalars
        else:
            output[key] = value

    return output

## Assembled Dataset

In [5]:
dataset_name = "OpenFF Organometallic Complexes Architector Minimum Energy Structures v0.0"
tagline = "BP86/def2-TZVP single metal complex optimizations with Pd, Fe, Zn, Cu, Li, and Mg and charge of {-1,0,+1} and multiplicities ranging from 1 to 6 and coordination ranging from 1 to 12, and MW <= 1005 Da."
description = ("""
This dataset was generated using [architector](https://github.com/lanl/Architector/tree/Secondary_Solvation_Shell), the
details of the HDF5 file can be found in the Zenodo record (https://zenodo.org/records/19372923). This dataset contains 
40,486 unique systems/configurations below 1005 Da using the same keys as in the HDF5 as entry labels.  The molecules 
are limited to containing transition metals Pd, Zn, Fe, Cu, Li, or Mg and also only contain elements Br, C, H, P, S, O,
N, F, Cl, or Br with overall charges: {-1,0,+1}. The metal coordination for each metal center is between 1 and 12. Each
molecule was preprocessed using gfn2-xtb as implemented in the architector package. This optimization dataset was then
run with the BP86/def2-TZVP. Each configuration is reported with the following properties: 'energy', 'gradient',
'dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'dipole_polarizabilities', 
'mulliken_charges'.
""")

dataset = client.add_dataset( # https://docs.qcarchive.molssi.org/user_guide/qcportal_reference.html
    "optimization", # collection type
    dataset_name, # Dataset name
    tagline=tagline,
    description=description,
    tags=["openff"],
    provenance={
        "qcportal": qcportal.__version__,
    },
    default_tag="openff",
    extras={
        "submitter": "jaclark5",
        "creation_date": date.today(),
        'collection_type': 'OptimizationDataset',
        "long_description": description,
        'long_description_url': f'https://github.com/openforcefield/qca-dataset-submission/tree/master/submissions/2026-04-06-{dataset_name.replace(" ", "-")}',
        "short_description": tagline,
        "dataset_name": dataset_name,
    },
)

In [6]:
hdf5_mapping = {
    "symbols": (get_symbols, "atomic_numbers"), 
    "geometry": "positions",
    "molecular_charge": "total_charge",
    "molecular_multiplicity": "spin_multiplicities",
    "identifiers": {"molecular_formula": (get_molecular_formula, "atomic_numbers"),},
    "extras": {'molecular_weight': (get_molecular_weight, "atomic_numbers")},
}

elements, molecular_weights, charges, multiplicities, metals, coord_ns, ox_states = [], [], [], [], [], [], []
conformers = Counter()
count_molecules = 0

errors = defaultdict(list)
errors_struc = defaultdict(list)
errors_mult = []
errors_misc = defaultdict(lambda: defaultdict(list))
failed_metals = defaultdict(lambda: 0)

hdf5 = h5py.File(f"architector_filtered.hdf5", 'r')
for ii, (label, mol_hdf5) in enumerate(hdf5.items()):
    mol_dict = convert_hdf5_group(mol_hdf5)
    lx = 1
    metals.append(mol_dict["metal"])
    coord_ns.append(mol_dict["coordination_number"])
    ox_states.append(mol_dict["oxidation_state"])
    
    ## Decide to filter
    try:
        input = apply_mapping(hdf5_mapping, mol_dict, index=0)
        if len(input["symbols"]) != np.shape(input["geometry"])[0]:
            raise ValueError(f"Geometries don't match number of symbols: {len(input['symbols'])} != {np.shape(input['geometry'])[0]}")
    except Exception as e:
        errors_struc[str(e)[:30]].append([label, 1, str(e)])
        continue

    input["geometry"] *= 10 / constants.bohr2angstroms # Convert from nm to Bohr (a0)

    try:
        molecule = Molecule(
            name=label,
            fix_com=True,
            fix_orientation=True,
            fix_symmetry="c1",
            **input
        )
        dataset.add_entry(name=label, initial_molecule=molecule)
        count_molecules += 1
        conformers[label] += 1
    except Exception as e:
        if "Inconsistent or unspecified chg/mult" in str(e):
            errors_mult.append(label)
        else:
            errors_misc[str(e)[:30]][label].append(str(e))
        continue

    elements.extend(list(set(input['symbols'])))
    molecular_weights.append(input['extras']["molecular_weight"])
    charges.append(input["molecular_charge"])
    multiplicities.append(input["molecular_multiplicity"])

dataset.extras["elements"] = sorted(list(set(elements)))

In [7]:
print(f"Number of molecules removed for unspecified chg/mult: {len(errors_mult)}")
print(f"Number of molecules removed for structure issues: {len(errors_struc)}")
print(f"Number of conformers accepted: {len(dataset.entry_names)}")

Number of molecules removed for unspecified chg/mult: 0
Number of molecules removed for structure issues: 0
Number of conformers accepted: 40486


In [8]:
print(f"{len(dataset.entry_names)} conformers were imported.")

print("\nThe following errors DO remove molecules from the dataset:")
for err, values in errors_misc.items():
    print(f"    {len(values)}: '{err}'")

print(f"\nThere were {sum([len(x) for x in errors.values()])} molecules of {len(dataset.entry_names)} that failed to create SMILES.")

40486 conformers were imported.

The following errors DO remove molecules from the dataset:

There were 0 molecules of 40486 that failed to create SMILES.


In [9]:
errors_misc['OptimizationDataset.add_entry(']

defaultdict(list, {})

In [10]:
spec = QCSpecification(
    program='psi4',
    driver=SinglepointDriver.gradient,
    method='BP86',
    basis='def2-TZVP',
    keywords={
        'maxiter': 500, 
        'scf_properties': ['dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'mulliken_charges'],
        'function_kwargs': {'properties': ['dipole_polarizabilities']},
    },
    protocols={'wavefunction': 'none'}
)
opt_spec = OptimizationSpecification(
    program="geometric",
    qc_specification=spec, 
    keywords={
        "tmax": 0.3,
        "check": 0,
        "qccnv": False,
        "reset": True,
        "trust": 0.1,
        "molcnv": False,
        "enforce": 0.0,
        "epsilon": 1e-05,
        "maxiter": 300,
        "coordsys": "dlc",
        "convergence_set": "GAU"
    },
)
dataset.add_specification(name="BP86/def2-TZVP", specification=opt_spec)

InsertMetadata(error_description=None, errors=[], inserted_idx=[0], existing_idx=[])

In [11]:
scaffold.to_json(dataset, compress=True)
#dataset.submit()

## Make Outputs

In [12]:
print("Elements:", ", ".join(dataset.extras["elements"]))
print("Charges:", sorted(set([float(x) for x in charges])))
print("Multiplicities:", sorted(set(multiplicities)))
print("Metals:", Counter(metals))
print("Coordination Numbers:", Counter(coord_ns))
print("Oxidation States:", Counter(ox_states))

print("Molecular Weight (min mean max):", int(np.min(molecular_weights)), int(np.mean(molecular_weights)), int(np.max(molecular_weights)))
            
print("Number of Molecules:", len(conformers))
print("Number of Conformers:", sum(conformers.values()))
n_conformers = np.array(list(conformers.values()))
print("Number of conformers (min mean max):", int(np.min(n_conformers)), int(np.mean(n_conformers)), int(np.max(n_conformers)))

Elements: Br, C, Cu, Fe, H, Li, Mg, N, O, P, Pd, S, Zn
Charges: [-1.0, 0.0, 1.0]
Multiplicities: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]
Metals: Counter({b'Fe': 13084, b'Pd': 8444, b'Cu': 8163, b'Mg': 4928, b'Zn': 4434, b'Li': 1433})
Coordination Numbers: Counter({8: 7460, 6: 5913, 7: 5669, 4: 4983, 3: 4933, 5: 4774, 9: 3199, 2: 2416, 10: 715, 12: 238, 1: 186})
Oxidation States: Counter({1: 9482, 2: 8350, 0: 5846, 3: 5180, 4: 5067, 5: 3344, 6: 1672, 7: 1545})
Molecular Weight (min mean max): 22 392 1005
Number of Molecules: 40486
Number of Conformers: 40486
Number of conformers (min mean max): 1 1 1


In [13]:
for spec, obj in dataset.specifications.items():
    obj = obj.dict()
    obj = obj['specification']
    print(obj.keys())
    print("* Spec:", spec)
    print(f"    * program: {obj['program']}")
    print(f"    * keywords:")
    for k, v in obj["keywords"].items():
        print(f"       * {k}: {v}")
    print(f"    * qc_specification:")
    for k, field in obj['qc_specification'].items():
        print(f"       * {k}: {field}")
    print("* SCF properties:")
    for field in obj['qc_specification']['keywords']["scf_properties"]:
        print(f"       * {field}")

dict_keys(['program', 'qc_specification', 'keywords', 'protocols'])
* Spec: BP86/def2-TZVP
    * program: geometric
    * keywords:
       * tmax: 0.3
       * check: 0
       * qccnv: False
       * reset: True
       * trust: 0.1
       * molcnv: False
       * enforce: 0.0
       * epsilon: 1e-05
       * maxiter: 300
       * coordsys: dlc
       * convergence_set: GAU
    * qc_specification:
       * program: psi4
       * driver: SinglepointDriver.deferred
       * method: bp86
       * basis: def2-tzvp
       * keywords: {'maxiter': 500, 'scf_properties': ['dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'mulliken_charges'], 'function_kwargs': {'properties': ['dipole_polarizabilities']}}
       * protocols: {'wavefunction': <WavefunctionProtocolEnum.none: 'none'>, 'stdout': True, 'error_correction': {'default_policy': True, 'policies': None}, 'native_files': <NativeFilesProtocolEnum.none: 'none'>}
* SCF properties:
       * dipole
       * quad